In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings('ignore')

In [2]:
import sys
sys.path.append('../')
from src import dataloader
from model import configs, engine
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
data_loader_training, data_loader_validate, data_loader_test = dataloader.dataloaders(
    batch_size = 1, 
    img_size = 16,
    in_channels = 8, 
    resmapling_status = False,
    data = 's2',
    exp_name = 'test'
    )

(13875, 36) | (8233, 36) | (12443, 36)


In [4]:
# # Create model configuration using custom configs module
config = configs.Configs(
    img_size = 16, 
    patch_size = 8, 
    embed_dim = 768, 
    mlp_dim = 512, 
    pool = 'cls',
    in_channels = 8,
    out_channels = 1, 
    num_heads = 8, 
    num_layers = 6, 
    cond = False,
    multi_conv = False,
    attn_dropout = 0.1, 
    proj_dropout = 0.1, 
    drop_path = 0.0,
    post_norm = False, 
    vis = True, 
    tokenizer = 'h',
    mask_modality = None
    ).call()

In [6]:
index = 1000
sample =  data_loader_training.dataset[index]
text = sample['EmbText']
text

'LIV_003 is MALVASIA_BIANCA vines on 4WIREWO trellises type, 12m row spacing and 7m canopy space.'

In [7]:
from model import attention

text_embed = attention.TextEmbed(config).to(device)
text_transformer = attention.TextEncoder(config.embed_dim, 
                config.num_layers, 
                config.num_heads, 
                dim_head = 96, 
                mult=4, 
                dropout=config.proj_dropout).to(device)

In [12]:
context = text_embed(text)
context.shape

torch.Size([96, 250, 768])

In [13]:
context = text_transformer(context)

In [18]:
print(context[0].shape, len(context[1]), context[1][0].shape)

torch.Size([96, 250, 768]) 6 torch.Size([96, 8, 250, 250])
